# Visualize Results
Notebook to load saved .npz results and plot simple diagnostics.

In [ ]:
import numpy as np
from sf_recon.utils.io import load_npz
print('Ready to load results: call load_npz with path')

In [ ]:
# ==============================================================================
# Velocity fields visualization (Cylinder)
# Line 1: vn-GT、vs-GT
# Line 2: vn-Recon、vs-Recon
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, patches
from IPython.display import HTML, display
import matplotlib as mpl                                                                                              
from matplotlib.colors import ListedColormap

bmap=mpl.cm.twilight_shifted 
rmap=mpl.cm.twilight
bluecolors=bmap(np.linspace(0,1,256)) 
redcolors=rmap(np.linspace(0,1,256)) 
cblue=ListedColormap(bluecolors[:100]) 
cred=ListedColormap(redcolors[128:228]) 

# Load result
with np.load('../data/simulation/cyl_v0_recon.npz', allow_pickle=True) as data:
    # vn fields
    v_gt_u = data.get('v_gt_u')
    v_gt_v = data.get('v_gt_v')
    v_recon_u = data.get('v_recon_u')
    v_recon_v = data.get('v_recon_v')
    # vs fields
    vs_gt_u = data.get('vs_gt_u')
    vs_gt_v = data.get('vs_gt_v')
    vs_recon_u = data.get('vs_recon_u')
    vs_recon_v = data.get('vs_recon_v')
    gt_pts = data.get('gt')
    recon_pts = data.get('recon')

def as_time_array(x):
    if x is None:
        return None
    try:
        a = np.array(x)
    except Exception:
        try:
            a = np.array(list(x))
        except Exception:
            return None
    if a.dtype == object:
        try:
            a = np.stack([np.asarray(e, dtype=float) for e in a])
        except Exception:
            return None
    try:
        return a.astype(float)
    except Exception:
        return None

v_gt_u = as_time_array(v_gt_u)
v_gt_v = as_time_array(v_gt_v)
v_recon_u = as_time_array(v_recon_u)
v_recon_v = as_time_array(v_recon_v)
vs_gt_u = as_time_array(vs_gt_u)
vs_gt_v = as_time_array(vs_gt_v)
vs_recon_u = as_time_array(vs_recon_u)
vs_recon_v = as_time_array(vs_recon_v)
gt_pts = as_time_array(gt_pts)
recon_pts = as_time_array(recon_pts)

if v_gt_u is None and v_recon_u is None and vs_gt_u is None and vs_recon_u is None:
    print('Velocity fields not found in NPZ.')
else:
    # --- Domain parameters ---
    Lx, Ly = 0.02, 0.2
    Wy = 0.02
    Radius = 0.004
    y_min, y_max = Ly / 2 - Wy, Ly / 2 + Wy

    sample = v_gt_u if v_gt_u is not None else (v_recon_u if v_recon_u is not None else (vs_gt_u if vs_gt_u is not None else vs_recon_u))
    T, d1, d2 = sample.shape

    # Auto-detect H/W orientation
    if d1 > d2:
        H, W = d1, d2
        is_transposed = False
    else:
        H, W = d2, d1
        is_transposed = True

    x = np.linspace(0, Lx, W)
    y = np.linspace(0, Ly, H)
    X, Y = np.meshgrid(x, y)
    y_idx = np.where((y >= y_min) & (y <= y_max))[0]

    def get_local_min_max(u_field, v_field, y_indices, transposed):
        if u_field is None or v_field is None:
            return 0.0, 1.0
        speed_full = np.sqrt(u_field**2 + v_field**2)
        if not transposed:
            local_data = speed_full[:, y_indices, :]
        else:
            local_data = speed_full[:, :, y_indices]
        _min, _max = np.nanmin(local_data), np.nanmax(local_data)
        return _min, _max

    vmin_vn, vmax_vn = get_local_min_max(v_gt_u, v_gt_v, y_idx, is_transposed)
    vmin_vs, vmax_vs = get_local_min_max(vs_gt_u, vs_gt_v, y_idx, is_transposed)
    vmin_vn_re, vmax_vn_re = get_local_min_max(v_recon_u, v_recon_v, y_idx, is_transposed)
    vmin_vs_re, vmax_vs_re = get_local_min_max(vs_recon_u, vs_recon_v, y_idx, is_transposed)

    fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    # cmap = plt.cm.get_cmap('viridis')

    init_vn = np.zeros((H, W))
    init_vs = np.zeros((H, W))
    im0 = axs[0, 0].imshow(init_vn, origin='lower', extent=[0, Lx, 0, Ly], cmap=cred, vmin=vmin_vn, vmax=vmax_vn, interpolation='bilinear')
    im1 = axs[0, 1].imshow(init_vs, origin='lower', extent=[0, Lx, 0, Ly], cmap=cblue, vmin=vmin_vs, vmax=vmax_vs, interpolation='bilinear')
    im2 = axs[1, 0].imshow(init_vn, origin='lower', extent=[0, Lx, 0, Ly], cmap=cred, vmin=vmin_vn_re, vmax=vmax_vn_re, interpolation='bilinear')
    im3 = axs[1, 1].imshow(init_vs, origin='lower', extent=[0, Lx, 0, Ly], cmap=cblue, vmin=vmin_vs_re, vmax=vmax_vs_re, interpolation='bilinear')

    obs_center = (Lx / 2, Ly / 2)
    for ax in axs.ravel():
        ax.add_patch(patches.Circle(obs_center, Radius, facecolor='gray', edgecolor='black', zorder=10))
        ax.set_xlim(0, Lx)
        ax.set_ylim(y_min, y_max)
        ax.set_aspect('equal')
        ax.set_ylabel('y [m]')
        ax.set_xlabel('x [m]')

    fig.colorbar(im0, ax=axs[0, 0], fraction=0.046, pad=0.04).set_label('vn speed (Local)')
    fig.colorbar(im1, ax=axs[0, 1], fraction=0.046, pad=0.04).set_label('vs speed (Local)')
    fig.colorbar(im2, ax=axs[1, 0], fraction=0.046, pad=0.04).set_label('vn speed (Local)')
    fig.colorbar(im3, ax=axs[1, 1], fraction=0.046, pad=0.04).set_label('vs speed (Local)')

    s0, s1, s2, s3, scatter0, scatter1 = [None], [None], [None], [None], [None], [None]

    def _remove_streamplot(sp):
        if sp is None:
            return
        try:
            sp.lines.remove()
        except Exception:
            pass
        try:
            for a in sp.arrows:
                a.remove()
        except Exception:
            pass

    def update(i):
        # vn GT
        if v_gt_u is not None:
            u, v = v_gt_u[i], v_gt_v[i]
            if is_transposed:
                u, v = u.T, v.T
            im0.set_data(np.sqrt(u**2 + v**2))
            _remove_streamplot(s0[0])
            s0[0] = axs[0, 0].streamplot(X, Y, u, v, color='whitesmoke', density=7.5, linewidth=0.7)
        # vs GT
        if vs_gt_u is not None:
            u2, v2 = vs_gt_u[i], vs_gt_v[i]
            if is_transposed:
                u2, v2 = u2.T, v2.T
            im1.set_data(np.sqrt(u2**2 + v2**2))
            _remove_streamplot(s1[0])
            s1[0] = axs[0, 1].streamplot(X, Y, u2, v2, color='whitesmoke', density=7.5, linewidth=0.7)
        # vn Recon
        if v_recon_u is not None:
            ur, vr = v_recon_u[i], v_recon_v[i]
            if is_transposed:
                ur, vr = ur.T, vr.T
            im2.set_data(np.sqrt(ur**2 + vr**2))
            _remove_streamplot(s2[0])
            s2[0] = axs[1, 0].streamplot(X, Y, ur, vr, color='whitesmoke', density=7.5, linewidth=0.7)
        # vs Recon
        if vs_recon_u is not None:
            u2r, v2r = vs_recon_u[i], vs_recon_v[i]
            if is_transposed:
                u2r, v2r = u2r.T, v2r.T
            im3.set_data(np.sqrt(u2r**2 + v2r**2))
            _remove_streamplot(s3[0])
            s3[0] = axs[1, 1].streamplot(X, Y, u2r, v2r, color='whitesmoke', density=7.5, linewidth=0.7)

        # Particles (GT top row, recon bottom row)
        if gt_pts is not None:
            pts = np.asarray(gt_pts[i])
            mask = (pts[:, 1] >= y_min) & (pts[:, 1] <= y_max)
            vis_pts = pts[mask]
            if scatter0[0] is not None:
                scatter0[0].remove()
            scatter0[0] = axs[0, 0].scatter(vis_pts[:,0], vis_pts[:,1], s=0, facecolors='lightblue', edgecolors='white', zorder=5)
        if recon_pts is not None:
            pts = np.asarray(recon_pts[i])
            mask = (pts[:, 1] >= y_min) & (pts[:, 1] <= y_max)
            vis_pts = pts[mask]
            if scatter1[0] is not None:
                scatter1[0].remove()
            scatter1[0] = axs[1, 0].scatter(vis_pts[:,0], vis_pts[:,1], s=0, facecolors='lightblue', edgecolors='white', zorder=5)

        axs[0, 0].set_title(f'vn GT (Local Norm) t={i}')
        axs[0, 1].set_title(f'vs GT (Local Norm) t={i}')
        axs[1, 0].set_title(f'vn Recon (Local Norm) t={i}')
        axs[1, 1].set_title(f'vs Recon (Local Norm) t={i}')
        return im0, im1, im2, im3

    num_frames = min(10, T)
    anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=250, blit=False)
    plt.close(fig)
    anim.save("../data/simulation/cyl_v0_recon.gif")
    # display(HTML(anim.to_jshtml()))